### **TAYLOR SWIFT**
---
##### Trabajo practico II - Procesamiento del lenguaje natural - ITBA
##### *GPT2.*
---
Miembros del grupo:
* Magdalena Eppens - 62450
* Sofía Hanna Feilbogen - 61889
* Sofía Gonzalez del Solar - 62292
* Nicole Reiman - 62407


### 1. Librerias

In [2]:
!pip install datasets
!pip install accelerate
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import transformers
import accelerate
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Bases de datos

In [5]:
folder_path = '/content/drive/MyDrive/TP NLP/TP2/Bases de datos'
all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
df_list = [pd.read_csv(file) for file in all_files]
df = pd.concat(df_list, ignore_index=True)

# Combinar las letras de cada canción en una sola fila
df_combined = df.groupby(['album_name', 'track_title', 'track_n'])['lyric'].apply(' '.join).reset_index()

# Pasamos el df a csv
df_combined.to_csv('/content/drive/MyDrive/TP NLP/TP2/DatasetEntero.csv', index=False)
excluded_songs = ["Miss Americana & The Heartbreak Prince", "Forever & Always", "Mr. Perfectly Fine"]
df_combined = df_combined[~df_combined['track_title'].isin(excluded_songs)]

### PASO 3: entrenamiento y testeo

In [2]:
!pip uninstall -y transformers accelerate
!pip install -U transformers accelerate

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: accelerate 0.31.0
Uninstalling accelerate-0.31.0:
  Successfully uninstalled accelerate-0.31.0
  Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
  Using cached accelerate-0.31.0-py3-none-any.whl (309 kB)


In [4]:
from datasets import Dataset, DatasetDict
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split



train_texts, val_texts = train_test_split(df_combined['lyric'], test_size=0.2, random_state=42)

train_dataset = Dataset.from_dict({'text': train_texts})
val_dataset = Dataset.from_dict({'text': val_texts})
datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=512)

tokenized_datasets = datasets.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_datasets = tokenized_datasets.map(lambda examples: {'labels': examples['input_ids']}, batched=True)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=500,
)

model = GPT2LMHeadModel.from_pretrained('gpt2')

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
)

trainer.train()


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step,Training Loss
10,2.975300
20,2.843000
30,2.750600
40,3.001200
50,2.699200
60,2.924200
70,2.830200
80,2.652700
90,2.835900
100,2.789200


TrainOutput(global_step=195, training_loss=2.7274915939722306, metrics={'train_runtime': 63.2324, 'train_samples_per_second': 6.12, 'train_steps_per_second': 3.084, 'total_flos': 101120016384000.0, 'train_loss': 2.7274915939722306, 'epoch': 3.0})

### PASO 4: generación del texto

In [6]:
# Funcion para generar el texto.
# Hay 15 palabras que son de promt, y 100 generadas
def generate_text(prompt, max_length=115):
    inputs = tokenizer.encode(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(inputs, max_length=max_length, num_return_sequences=1, no_repeat_ngram_size=2, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Se corre la funcion
prompt = "You know I adore you, I'm crazier for you Than I was at 16, ·"
generated_text = generate_text(prompt, max_length=115)

print("El promp ingresado fue:",prompt)
print("")
print("El texto generado fue:",generated_text)

El promp ingresado fue: You know I adore you, I'm crazier for you Than I was at 16, ·

El texto generado fue: You know I adore you, I'm crazier for you Than I was at 16, · I've been waiting for this for so long, but I can't wait for it to be over, so I'll be the one who's gonna be there, and I know you're gonna love me, too, you know that, right? I love you too I want you to know how I feel, when I see you in the mirror, in my mind, my, your, mine, yours, me I wanna be with you I like you so much, that
